In [ ]:
! nvidia-smi

In [ ]:
from datasets import load_dataset
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
import gc
import time
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig,SFTTrainer
from PIL import Image

In [ ]:
from huggingface_hub import HfFolder, whoami

#Replace 'your_token' with your actual Hugging Face token
hf_token = 'YOUR_HF_TOKEN'  #Write token for Granite-Vision-3.2 Model
HfFolder.save_token(hf_token)

user = whoami()
print(f"Logged in as {user['name']}")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
system_message = """You are a Vision-Language Model specialized in interpreting PCB circuit designs and waveform simulation graphs. Your task is to analyze the provided image, determine whether it is a circuit schematic, PCB layout, or waveform simulation graph, and generate a structured, technically detailed description.

For circuit schematics and PCB layouts, identify the circuit topology, functional blocks, component details, signal flow, electrical parameters, and PCB layout considerations. For waveform simulation graphs, extract key waveform characteristics, analyze signal integrity, and assess circuit performance.

Focus on delivering precise, structured, and insightful technical descriptions based on visual information. Avoid unnecessary elaboration beyond what is essential for accuracy."""

In [ ]:
def load_image(image_path, target_size=(512, 512)):
    with Image.open(image_path) as img:
        img = img.convert("RGB")  # Ensure compatibility
        img = img.resize(target_size, Image.LANCZOS) #High-Quality Downscaling
        return img.copy()

def format_data(sample):
    return [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": load_image(sample["image"]),
                },
                {
                    
                    "type": "text",
                    "text": sample["conversations"][0]["value"],
                }
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": sample["conversations"][1]["value"]
                }
            ],
        },
    ]

In [ ]:
raw_datasets = load_dataset("json", data_files="json_data_12_03_2025_final.json")
split_datasets = raw_datasets["train"].train_test_split(test_size=0.2)
train_dataset = split_datasets["train"] 
eval_dataset = split_datasets["test"]

print(f"Training Dataset Size: {len(train_dataset)}") 
print(f"Evaluation Dataset Size: {len(eval_dataset)}")

In [ ]:
train_dataset[0]

In [ ]:
train_dataset_formatted = [format_data(sample) for sample in train_dataset]
eval_dataset_formatted = [format_data(sample) for sample in eval_dataset]

In [ ]:
train_dataset_formatted[0]

In [ ]:
# Save train_dataset and eval_dataset to disk
train_dataset_path = "./datasets/train_dataset_13_03_2025_granite"
eval_dataset_path = "./datasets/eval_dataset_13_03_2025_granite"

train_dataset.save_to_disk(train_dataset_path)
eval_dataset.save_to_disk(eval_dataset_path)

print(f"Train dataset saved to: {train_dataset_path}")
print(f"Eval dataset saved to: {eval_dataset_path}")

In [ ]:
#Remove Model and Clean GPU

def clear_memory():
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'peft_model' in globals(): del globals()['peft_model']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory() #Clearing variable, model and GPU

In [ ]:
model_id = "ibm-granite/granite-vision-3.2-2b"

In [ ]:
# BitsAndBytesConfig int-4 config

bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=["vision_tower", "lm_head"],  # Skip problematic modules
        #llm_int8_enable_fp32_cpu_offload=True
    )
# Load model and tokenizer
model = AutoModelForVision2Seq.from_pretrained(
                                model_id,
                                device_map="auto",
                                torch_dtype=torch.bfloat16,
                                quantization_config=bnb_config,
        ).to("cuda")

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
# x = [name for name, _ in model.named_modules() if 'language_model' in name and '_proj' in name]
# x

In [ ]:
# Configure LoRA

# peft_config = LoraConfig(
#         r=8,
#         lora_alpha=8,
#         lora_dropout=0.1,
#         target_modules=[name for name, _ in model.named_modules() if 'language_model' in name and '_proj' in name],
#         use_dora=True,
#         init_lora_weights="gaussian"
#     )

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

model.add_adapter(peft_config)
model.enable_adapters()
model = get_peft_model(model, peft_config)

# Print trainable parameters
model.print_trainable_parameters()

In [ ]:
# Configure training arguments using SFTConfig

training_args = SFTConfig(
    output_dir="granite-vision-3.2-2b-13-03-2025", 
    logging_dir="granite-vision-3.2-2b-13-03-2025/logs",
    num_train_epochs=1,  
    per_device_train_batch_size=4,  
    gradient_accumulation_steps=4, 
    remove_unused_columns = False,
    optim="adamw_torch_fused",  
    learning_rate=2e-4,  
    lr_scheduler_type="constant",
    logging_steps=10, 
    save_strategy="steps", 
    save_steps=25,  
    save_total_limit=10,
    metric_for_best_model="eval_loss",  
    greater_is_better=False, 
    bf16=True,  
    max_grad_norm=0.3,  
    warmup_ratio=0.03, 
    push_to_hub=True, 
    report_to="tensorboard",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="",  
    dataset_kwargs={"skip_prepare_dataset": True},
)

In [ ]:
def collate_fn(examples):
    texts = [processor.apply_chat_template(example, tokenize=False) for example in examples]

    image_inputs = []
    for example in examples:
      image = example[1]['content'][0]['image']
      if image.mode != 'RGB':
          image = image.convert('RGB')
      image_inputs.append([image])

    batch = processor(text=texts, images=image_inputs, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    assistant_tokens = processor.tokenizer("<|assistant|>", return_tensors="pt")['input_ids'][0]
    eos_token = processor.tokenizer("<|end_of_text|>", return_tensors="pt")['input_ids'][0]

    for i in range(batch["input_ids"].shape[0]):
        apply_loss = False
        for j in range(batch["input_ids"].shape[1]):
            if not apply_loss:
                labels[i][j] = -100
            if ((j>=len(assistant_tokens)+1) and
                torch.all(batch["input_ids"][i][j+1-len(assistant_tokens):j+1]==assistant_tokens)):
                apply_loss = True
            if batch["input_ids"][i][j]==eos_token:
                apply_loss = False

    batch["labels"] = labels

    return batch

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    data_collator=collate_fn,
    peft_config=peft_config,
    tokenizer=processor.tokenizer,
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir granite-vision-3.2-2b-13-03-2025/logs --port=6007


In [ ]:
trainer.train()

In [ ]:
trainer.save_model(training_args.output_dir)

In [ ]:
trainer.push_to_hub()

In [ ]:
clear_memory()